# Wallet 0xbe90db Transfer History

Notebook to explore 2024 SUI transfers for `0xbe90dbe25c747f23e7b93f8948f240a90aa106597b7dab0b33e852b0b1950aa8` and highlight transfers to designated sale addresses.

In [ ]:
import json
from pathlib import Path
from datetime import datetime, timezone
from typing import Any, Dict, Iterable, List

import matplotlib.pyplot as plt
import pandas as pd
import requests

plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

In [ ]:
RPC_URL = "https://fullnode.mainnet.sui.io/"
WALLET_ADDRESS = "0xbe90dbe25c747f23e7b93f8948f240a90aa106597b7dab0b33e852b0b1950aa8"
SALE_ADDRESSES = {
    "0x457fd3e8295543c23cfc2182b33247338cb04ddf43d0afa02a34591fd67415e7",
    "0x56f173b16adc96d0a8ac7e3f21e05c3a6621726ab8020140867340198b86e75c",
    "0xbe9e8a62a74d1246924e10c46b3de0a22ec20998f66d1562c615d3ee846e922e",
    "0x7f6626f0787309355b4b85de40596f5b096050eab5b9061dcb9001df961b3bfb",
}
DATA_PATH = Path("../data/wallet_0xbe90db_transfer_history/transactions_2024.json")
SUI_DECIMALS = 10 ** 9
START = datetime(2024, 1, 1, tzinfo=timezone.utc)
END = datetime(2025, 1, 1, tzinfo=timezone.utc)

In [ ]:
def fetch_transactions(direction: str, page_size: int = 100) -> List[Dict[str, Any]]:
    cursor: Dict[str, str] | None = None
    results: List[Dict[str, Any]] = []
    while True:
        payload = {
            "jsonrpc": "2.0",
            "id": 1,
            "method": "suix_queryTransactionBlocks",
            "params": [
                {
                    "filter": {f"{direction}Address": WALLET_ADDRESS},
                    "options": {
                        "showBalanceChanges": True,
                        "showEffects": True,
                        "showInput": True,
                        "showEvents": True,
                    },
                },
                cursor,
                page_size,
                True,
            ],
        }
        response = requests.post(RPC_URL, json=payload, timeout=60)
        response.raise_for_status()
        result = response.json()["result"]
        data = result["data"]
        if not data:
            break
        for tx in data:
            ts = int(tx["timestampMs"])
            if ts < int(START.timestamp() * 1000):
                return results
            if ts < int(END.timestamp() * 1000):
                results.append(tx)
        if not result.get("hasNextPage"):
            break
        cursor = result.get("nextCursor")
        if cursor is None:
            break
    return results


def load_transactions(refresh: bool = False) -> Dict[str, Any]:
    if refresh or not DATA_PATH.exists():
        print("Fetching from Sui RPC (this can take ~1 minute)...")
        from_transactions = fetch_transactions("From")
        to_transactions = fetch_transactions("To")
        payload = {
            "wallet": WALLET_ADDRESS,
            "range": {
                "start_ms": int(START.timestamp() * 1000),
                "end_ms": int(END.timestamp() * 1000),
            },
            "from_transactions": from_transactions,
            "to_transactions": to_transactions,
        }
        DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
        DATA_PATH.write_text(json.dumps(payload, indent=2))
        print(f"Saved dataset to {DATA_PATH}")
        return payload
    print(f"Loading cached dataset from {DATA_PATH}")
    return json.loads(DATA_PATH.read_text())


data = load_transactions(refresh=False)
len(data["from_transactions"]), len(data["to_transactions"])

In [ ]:
def normalize_transactions(transactions: Iterable[Dict[str, Any]], direction: str) -> List[Dict[str, Any]]:
    rows: List[Dict[str, Any]] = []
    for tx in transactions:
        ts = pd.to_datetime(int(tx["timestampMs"]), unit="ms", utc=True)
        balance_changes = tx.get("balanceChanges") or []
        sale_recipients: List[str] = []
        sale_amount_mist = 0
        wallet_change_mist = 0

        for change in balance_changes:
            owner_info = change.get("owner", {})
            address = owner_info.get("AddressOwner")
            if not address:
                continue
            amount = int(change["amount"])
            if address.lower() == WALLET_ADDRESS.lower():
                wallet_change_mist += amount
            if (
                amount > 0
                and address in SALE_ADDRESSES
                and change.get("coinType") == "0x2::sui::SUI"
            ):
                sale_recipients.append(address)
                sale_amount_mist += amount

        rows.append(
            {
                "timestamp": ts,
                "digest": tx["digest"],
                "direction": direction,
                "wallet_change_sui": wallet_change_mist / SUI_DECIMALS,
                "sale_recipients": sale_recipients,
                "sale_count": len(sale_recipients) if direction == "outgoing" else 0,
                "sale_amount_sui": sale_amount_mist / SUI_DECIMALS,
            }
        )
    return rows


rows: List[Dict[str, Any]] = []
rows.extend(normalize_transactions(data["from_transactions"], "outgoing"))
rows.extend(normalize_transactions(data["to_transactions"], "incoming"))

transactions_df = pd.DataFrame(rows).drop_duplicates(subset=["digest", "direction"]).sort_values("timestamp").reset_index(drop=True)
transactions_df.head()

In [ ]:
sales_df = transactions_df[transactions_df["sale_count"] > 0].copy()
print(f"Identified {len(sales_df)} sale transactions")
if not sales_df.empty:
    weekly_sales = (
        sales_df.set_index("timestamp")["sale_count"].resample("W-MON").sum().rename("weekly_sales")
    )
    cumulative_sales = weekly_sales.cumsum().rename("cumulative_sales")
    display(pd.concat([weekly_sales, cumulative_sales], axis=1).fillna(0).head())
else:
    weekly_sales = pd.Series(dtype=float)
    cumulative_sales = pd.Series(dtype=float)


In [ ]:
if not sales_df.empty:
    fig, ax = plt.subplots(figsize=(12, 6))
    weekly_sales.plot(ax=ax, marker="o", label="Weekly sales")
    ax.set_title("Wallet 0xbe90db: Number of sales per week (2024)")
    ax.set_ylabel("Number of sales")
    ax.set_xlabel("Week starting")

    ax2 = ax.twinx()
    cumulative_sales.plot(ax=ax2, color="tab:red", linestyle="--", label="Cumulative sales")
    ax2.set_ylabel("Cumulative sales")

    lines, labels = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines + lines2, labels + labels2, loc="upper left")
    fig.tight_layout()
else:
    print("No sales detected for the selected period")

In [ ]:
if not sales_df.empty:
    counterparties = (
        sales_df.explode("sale_recipients").dropna(subset=["sale_recipients"])
        .groupby("sale_recipients")
        .agg(sales=("sale_count", "sum"), total_sui=("sale_amount_sui", "sum"))
        .sort_values("sales", ascending=False)
    )
    display(counterparties)
